# 13 — Centralised coalition safety-game shielding

This notebook introduces winning-region shielding for a coalition in a PettingZoo
Parallel environment. The base `ChickenMatrix` environment implements
`TabularParallelEnv`; the existing `LabelledParallelEnv` supplies proposition
labels; and `CoalitionLTLShield` builds and enforces the safety game.

We use the safety property

$$
\mathbf{G}\neg\mathit{crash}.
$$

The DFA's accepting state represents a **bad prefix**: a round in which both
players choose `straight`.

For a focal coalition $C$, MASA requires

$$
\exists a_C\;\forall a_{-C}\;\forall s'\in
\operatorname{Succ}(s,a_C,a_{-C}): (q',s')\in W.
$$

Actions of agents outside the coalition are unrestricted. The shield does not
observe an outsider's simultaneous action before choosing the coalition action.

In [1]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import sys

import numpy as np

# Run from the repository root or from notebooks/tutorials/.
HERE = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (HERE, *HERE.parents) if (candidate / "masa").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the MASA-Safe-RL source tree.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from masa.common.multi_agent import Coalition
from masa.deterministic_shield import CoalitionLTLShield, random_safe
from masa.envs.multiagent.matrix.chicken import Actions
from masa.examples.chicken_safety_game import (
    make_labelled_chicken_env,
    make_never_crash_dfa,
)

ACTION_NAMES = {
    int(Actions.Swerve): "swerve",
    int(Actions.Straight): "straight",
}


def action_name(action: int) -> str:
    return ACTION_NAMES[int(action)]


def make_shield(
    coalition,
    *,
    mode="preemptive",
    execution="centralised",
    replacement=None,
    max_moves=8,
):
    return CoalitionLTLShield(
        make_labelled_chicken_env(max_moves=max_moves),
        coalition=Coalition(tuple(coalition)),
        dfa=make_never_crash_dfa(),
        mode=mode,
        execution=execution,
        replacement=replacement,
    )

ModuleNotFoundError: No module named 'masa.envs.multiagent.base'

## Inspect the tabular and labelled stack

The finite transition support belongs to the environment. No separate game-model
object is passed to the shield.

In [ ]:
labelled = make_labelled_chicken_env(max_moves=4)
base = labelled.env

print(type(base).__name__)
print("Finite states:", base.n_states)
print("Full joint actions:", base.n_joint_actions)
print("Joint action 2:", base.decode_joint_action(2))
print("Successor from reset:", base.successors(0, (Actions.Straight, Actions.Swerve)))

assert base.n_states == 5
assert base.successors(0, (Actions.Straight, Actions.Swerve)) == (3,)
labelled.close()

## One focal player against an unrestricted opponent

Only `player_0` is shielded. `player_1` is outside the coalition, so `player_0`
must choose an action that is safe against either opponent action. Even when the
submitted opponent action is `swerve`, `player_0=straight` is not robustly safe,
because the same focal action would crash against `player_1=straight`.

In [ ]:
single = make_shield(("player_0",), mode="preemptive")
observations, infos = single.reset(seed=0)

print("Tabular state:", single.tabular_state)
print("Coalition action order:", single.coalition_actions)
print("Robust mask:", single.coalition_action_mask().tolist())
print("Safe actions:", single.safe_coalition_actions())

assert single.coalition_action_mask().tolist() == [True, False]
assert single.safe_coalition_actions() == ({"player_0": int(Actions.Swerve)},)

In [ ]:
try:
    single.step(
        {
            "player_0": int(Actions.Straight),
            "player_1": int(Actions.Swerve),
        }
    )
except ValueError as exc:
    print("Rejected before stepping:", exc)

# A robust focal action executes unchanged, whatever the opponent proposes.
_, _, _, _, infos = single.step(
    {
        "player_0": int(Actions.Swerve),
        "player_1": int(Actions.Straight),
    }
)
print("Executed focal action:", action_name(infos["player_0"]["shield_executed_action"]))
print("Observed labels:", infos["player_0"]["labels"])
assert "crash" not in infos["player_0"]["labels"]
single.close()

### Exhaustively check the universal opponent quantifier

For every action exposed by the singleton coalition mask, enumerate every legal
opponent action and every supported successor. Each resulting product state must
remain in the winning region after the DFA consumes the successor label.

In [ ]:
single = make_shield(("player_0",), mode="preemptive")
single.reset(seed=0)
base = single.tabular_env
state = single.tabular_state
q = single.automaton_state
dfa = make_never_crash_dfa()
q_index = {dfa_state: index for index, dfa_state in enumerate(dfa.states)}

for coalition_index in np.flatnonzero(single.coalition_action_mask()):
    focal = single.decode_coalition_action(int(coalition_index))["player_0"]
    for opponent in base.get_legal_actions(state, "player_1"):
        full_action = (focal, opponent)
        for successor in base.successors(state, full_action):
            labels = single.labels_for_state(successor)
            next_q = dfa.transition(q, labels)
            product_state = q_index[next_q] * base.n_states + successor
            assert single.winning_region[product_state]
            print(
                f"player_0={action_name(focal):8s}",
                f"player_1={action_name(opponent):8s}",
                "->", sorted(labels),
            )
single.close()

## Centralised execution of both players

When both agents are controlled centrally, the controller chooses a complete joint
action. Three tuples are safe; only `(straight, straight)` is blocked.

In [ ]:
team = make_shield(("player_0", "player_1"), mode="preemptive")
team.reset(seed=0)

for index, joint in enumerate(team.coalition_actions):
    names = tuple(action_name(action) for action in joint)
    print(index, names, "SAFE" if team.coalition_action_mask()[index] else "blocked")

assert team.coalition_action_mask().tolist() == [True, True, True, False]
team.close()

## Postposed joint-action replacement

Postposed mode preserves safe proposals. When the coalition proposes the blocked
tuple, `random_safe` selects a safe **coalition-action index** uniformly from the
robust relation.

In [ ]:
team = make_shield(
    ("player_0", "player_1"),
    mode="postposed",
    replacement=random_safe(seed=7),
)
team.reset(seed=0)

_, _, _, _, infos = team.step(
    {
        "player_0": int(Actions.Straight),
        "player_1": int(Actions.Straight),
    }
)
info = infos["player_0"]
print("Proposed:", tuple(action_name(a) for a in info["shield_proposed_coalition_action"]))
print("Executed:", tuple(action_name(a) for a in info["shield_executed_coalition_action"]))
print("Intervened:", info["shield_joint_intervened"])
print("Labels:", info["labels"])

assert info["shield_joint_intervened"]
assert "crash" not in info["labels"]
team.close()

## Takeaways

- `TabularParallelEnv` supplies finite states, legal actions, and complete joint
  transition support.
- `LabelledParallelEnv` supplies the propositions consumed by the safety DFA.
- A proper coalition is protected against **every** legal action of its complement.
- Centralised execution may use a non-Cartesian safe joint-action relation because
  one controller selects the entire coalition tuple.
- Preemptive and postposed modes enforce the same relation; they differ only in
  whether an unsafe proposal is rejected or replaced.